In [1]:
import pandas as pd
import numpy as np
from io import StringIO
import pmdarima as pm
import warnings
from sklearn.metrics import mean_absolute_percentage_error

In [2]:
data = pd.read_csv('cpi_data.csv')

In [3]:
data = data.set_index('Date')

In [4]:
data.head()

,Education,Energy,Food,Housing,Medical,Transportation
Date,,,,,,
2017-01-01,139.041,199.608,248.242,247.942,471.700,199.292
2017-02-01,138.796,198.195,248.791,248.693,474.546,199.147
2017-03-01,136.234,198.597,249.165,248.978,474.561,200.091
2017-04-01,135.781,202.869,249.739,249.514,473.582,202.389
2017-05-01,135.563,203.132,250.016,250.376,473.512,202.212


In [5]:
log_data = np.log(data)
log_data.dropna(inplace=True)

In [6]:
log_data.head()

,Education,Energy,Food,Housing,Medical,Transportation
Date,,,,,,
2017-01-01,4.934769,5.296355,5.514404,5.513195,6.156343,5.294771
2017-02-01,4.933005,5.289251,5.516613,5.516219,6.162359,5.294043
2017-03-01,4.914374,5.291278,5.518115,5.517365,6.162390,5.298772
2017-04-01,4.911043,5.312560,5.520416,5.519515,6.160325,5.310192
2017-05-01,4.909436,5.313856,5.521525,5.522964,6.160177,5.309317


In [7]:
date = '2025-06-01'
train_data = log_data.loc[:date]
test_data = log_data.loc[date:]

In [8]:
warnings.filterwarnings("ignore")

In [9]:
categories = log_data.columns

In [ ]:
def run_hyndman_khandakar(series, m=12):
    model = pm.auto_arima(
        series,
        start_p=0, start_q=0,       
        max_p=3, max_q=3,           
        d=None, max_d=2,            
        start_P=0, start_Q=0,       
        max_P=2, max_Q=2,           
        D=None, max_D=1,            
        m=m,                        
        seasonal=True,              
        test='adf',                 
        stepwise=True,              
        information_criterion='aic',
        trace=False,                
        error_action='ignore',      
        suppress_warnings=True
    )
    return model

In [ ]:
trained_models = {}

for category in categories:
    model = run_hyndman_khandakar(train_data[category], m=12)
    
    order = model.order
    s_order = model.seasonal_order
    order_str = f"({order[0]},{order[1]},{order[2]})({s_order[0]},{s_order[1]},{s_order[2]})[12]"
    
    trained_models[category] = model
    
    print(f"{category:<15} | {order_str:<25} | {model.aic():<10.2f}")

Education       | (0,1,1)(1,0,0)[12]        | -888.83   
Energy          | (0,1,1)(0,0,0)[12]        | -438.92   
Food            | (0,2,1)(2,0,0)[12]        | -902.70   
Housing         | (0,2,1)(2,0,0)[12]        | -1005.88  
Medical         | (0,2,2)(0,0,0)[12]        | -889.01   
Transportation  | (0,1,1)(0,0,0)[12]        | -587.79   


In [ ]:
for category in categories:
    cat_test_data = test_data[category]
    steps = len(cat_test_data)
    
    cat_train_data = train_data[category]

    
    model_obj = trained_models[category]
    eval_model = pm.ARIMA(
        order=model_obj.order, 
        seasonal_order=model_obj.seasonal_order,
        suppress_warnings=True
    ).fit(cat_train_data)
    
    forecast = np.exp(eval_model.predict(n_periods=steps))
    
    mape = mean_absolute_percentage_error(np.exp(cat_test_data), forecast) * 100
    
    print(f"{category:<15} | Test MAPE: {mape:.2f}%")

Education       | Test MAPE: 0.31%
Energy          | Test MAPE: 5.46%
Food            | Test MAPE: 0.23%
Housing         | Test MAPE: 0.90%
Medical         | Test MAPE: 0.95%
Transportation  | Test MAPE: 2.43%


In [ ]:
n_forecast = 6

final_models = {}
forecasts = {}
conf_ints = {}

for category in categories:
    order = trained_models[category].order
    seasonal_order = trained_models[category].seasonal_order

    final_model = pm.ARIMA(
        order=order,
        seasonal_order=seasonal_order,
        suppress_warnings=True
    ).fit(log_data[category])

    final_models[category] = final_model

    log_fc, log_ci = final_model.predict(n_periods=n_forecast, return_conf_int=True)

    forecasts[category] = np.exp(np.asarray(log_fc))
    conf_ints[category] = np.exp(np.asarray(log_ci))

In [ ]:
last_date = log_data.index[-1] 
last_date = pd.to_datetime(last_date)
future_dates = pd.date_range(start=last_date + pd.DateOffset(months=1), periods=n_forecast, freq='MS')

forecast_df = pd.DataFrame(forecasts, index=future_dates)
forecast_df.index.name = 'Date'
forecast_df

,Education,Energy,Food,Housing,Medical,Transportation
Date,,,,,,
2026-06-01,147.832160,352.982971,349.979209,359.807329,594.474435,303.091456
2026-07-01,147.938765,354.771905,350.964487,361.439020,595.545411,304.223578
2026-08-01,148.019602,356.569905,351.903792,362.702878,596.613458,305.359929
2026-09-01,148.197930,358.377018,352.920345,364.040323,597.678551,306.500525
2026-10-01,148.346527,360.193289,354.083020,365.239076,598.740669,307.645381
2026-11-01,148.516560,362.018765,354.686632,366.315233,599.799789,308.794514


In [16]:
forecast_df.to_csv('sarima_predictions.csv')

In [15]:
conf_ints

{'Education': array([[146.95295219, 148.71662842],
        [146.61369952, 149.27580689],
        [146.36531041, 149.69259136],
        [146.2687816 , 150.15252115],
        [146.17700531, 150.54824739],
        [146.13006925, 150.94202606]]),
 'Energy': array([[334.22783326, 372.79054922],
        [322.91515935, 389.77143252],
        [315.82575031, 402.57039501],
        [310.46696459, 413.68036332],
        [306.1140214 , 423.82640521],
        [302.43639334, 433.33933697]]),
 'Food': array([[348.22845284, 351.73876739],
        [347.9476837 , 354.00744726],
        [347.53271585, 356.3298452 ],
        [347.08123237, 358.85769139],
        [346.65934249, 361.66567424],
        [345.58230525, 364.03081024]]),
 'Housing': array([[358.73704949, 360.88080178],
        [359.45156293, 363.43746557],
        [359.67873983, 365.7524428 ],
        [359.85847989, 368.27076311],
        [359.78840585, 370.77232124],
        [359.49325009, 373.26667343]]),
 'Medical': array([[591.33207558, 597.